# Fiorell.IA Blitz Perfection - Cost-Safe Release

Questo notebook esegue training/eval fuori da Google Drive (`/content`) e sincronizza su Drive solo alla fine.
Usa `RUN_MODE = "train"` per addestrare, `RUN_MODE = "reuse"` per rivalutare uno ZIP gia' presente.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive/regulatory-insight-engine")
LOCAL_ROOT = Path("/content/regulatory-insight-engine-local")
LOCAL_ARTIFACT = Path("/content/fiorellia-runs/blitz_delivery_latest")
LOCAL_RELEASE = Path("/content/releases/blitz_release_latest")
FINAL_NAME = "fiorellia_behavior_BLITZ_RELEASE_20260527"
EXPECTED_VERSION = "20260527-cost-safe-v5"

# Modalita': "train" rilancia training; "reuse" rivaluta adapter zip gia presente su Drive.
RUN_MODE = "train"
OPEN_GRADIO_AFTER_GO = False


def wait_for_path(path: Path, label: str, kind: str = "file", attempts: int = 8, base_sleep: float = 1.5) -> Path:
    def ok() -> bool:
        if kind == "dir":
            return path.is_dir()
        if kind == "any":
            return path.exists()
        return path.is_file()

    for attempt in range(1, attempts + 1):
        if ok():
            size = path.stat().st_size if path.is_file() else None
            suffix = f" size={size}" if size is not None else ""
            print(f"OK {label}: {path}{suffix}")
            return path
        if attempt < attempts:
            delay = base_sleep * attempt
            print(f"{label} non ancora visibile: {path}; retry {attempt}/{attempts} tra {delay:.1f}s")
            time.sleep(delay)
    raise RuntimeError(f"{label} non disponibile dopo {attempts} tentativi: {path}")


def sync_dir(src: Path, dst: Path, label: str) -> None:
    if not src.exists():
        print(f"skip sync {label}: source missing {src}")
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    wait_for_path(dst, label, kind="dir")
    print(f"synced {label}: {dst}")


subprocess.run(["pkill", "-f", "fiorellia_app_colab.py"], check=False)
subprocess.run(["pkill", "-f", "gradio"], check=False)

for path in [LOCAL_ROOT, LOCAL_ARTIFACT, LOCAL_RELEASE, Path("/content/fiorellia_blitz_eval")]:
    shutil.rmtree(path, ignore_errors=True)

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_ARTIFACT.mkdir(parents=True, exist_ok=True)
LOCAL_RELEASE.mkdir(parents=True, exist_ok=True)

print("Cloning repo locally outside Drive...")
subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "--branch",
        "main",
        "https://github.com/TheGenesisAIStory/regulatory-insight-engine.git",
        str(LOCAL_ROOT),
    ],
    check=True,
)

script_path = LOCAL_ROOT / "fiorellia_blitz_perfection.py"
script_text = script_path.read_text(encoding="utf-8")
assert EXPECTED_VERSION in script_text, f"Script non aggiornato: manca {EXPECTED_VERSION}"
print("repo ready:", EXPECTED_VERSION)

print("Checking GPU...")
subprocess.run(["nvidia-smi"], check=False)

import torch

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO CUDA"
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("gpu:", gpu_name)
if not torch.cuda.is_available():
    raise RuntimeError("BLOCCANTE: CUDA non disponibile. Attiva runtime GPU in Colab.")

is_a100 = "A100" in gpu_name
if is_a100:
    runtime_args = ["--batch-size", "4", "--epochs", "3", "--max-new-tokens", "96"]
else:
    runtime_args = ["--no-require-a100", "--batch-size", "1", "--epochs", "2", "--max-new-tokens", "64"]
    print("Modalita' T4/fallback attiva: batch=1, epochs=2, no A100 gate.")

if RUN_MODE == "reuse":
    for candidate in [
        DRIVE_ROOT / "fiorellia-runs" / "blitz_delivery_latest" / f"{FINAL_NAME}.zip",
        DRIVE_ROOT / "releases" / "blitz_release_latest" / f"{FINAL_NAME}.zip",
        DRIVE_ROOT / "fiorellia-runs" / "final_delivery_latest" / f"{FINAL_NAME}.zip",
        DRIVE_ROOT / "releases" / "gold_release_latest" / f"{FINAL_NAME}.zip",
    ]:
        if candidate.exists():
            target = LOCAL_ARTIFACT / f"{FINAL_NAME}.zip"
            shutil.copy2(candidate, target)
            wait_for_path(target, "local adapter zip")
            print("adapter zip copied:", candidate, "->", target)
            break

cmd = [
    sys.executable,
    "-u",
    str(script_path),
    "--local-first",
    "--copy-verdict-to-repo",
    "--no-flash-attn",
    *runtime_args,
]
if RUN_MODE == "reuse":
    cmd += ["--reuse-existing-adapter", "--skip-deps"]
else:
    cmd += ["--install-deps"]
if OPEN_GRADIO_AFTER_GO:
    cmd += ["--launch-gradio"]

print("+", " ".join(cmd))
proc = subprocess.Popen(
    cmd,
    cwd=LOCAL_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end="")
return_code = proc.wait()
print("Fiorell.IA Blitz cost-safe return_code=", return_code)

print("Syncing final artifacts to Drive...")
sync_dir(LOCAL_ARTIFACT, DRIVE_ROOT / "fiorellia-runs" / "blitz_delivery_latest", "Drive blitz artifacts")
sync_dir(LOCAL_RELEASE, DRIVE_ROOT / "releases" / "blitz_release_latest", "Drive blitz release")

for src_rel, dst_rel in [
    ("fiorellia/eval/blitz_verdict.md", "fiorellia/eval/blitz_verdict.md"),
    ("fiorellia/eval/reports/final_release/blitz_summary.json", "fiorellia/eval/reports/final_release/blitz_summary.json"),
]:
    src = LOCAL_ROOT / src_rel
    dst = DRIVE_ROOT / dst_rel
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        wait_for_path(dst, dst_rel)
        print("synced repo output:", dst)

print("DONE")
print("Return code:", return_code)
print("Artifacts:", DRIVE_ROOT / "fiorellia-runs" / "blitz_delivery_latest")
print("Release:", DRIVE_ROOT / "releases" / "blitz_release_latest")

if return_code == 0:
    print("ESITO: gate Blitz positivo. Apri la cella demo solo se serve.")
elif return_code == 2:
    print("ESITO: NO-GO reale, ma training/eval completati e salvati.")
else:
    print("ESITO: BLOCCO reale. Controlla eventuali trace locali:")
    print(LOCAL_ARTIFACT / "uncaught_jsondecode_traceback.txt")
    print(LOCAL_ARTIFACT / "uncaught_jsondecode.json")


## Demo Gradio opzionale

Esegui questa cella solo dopo training/eval, altrimenti la sessione resta aperta inutilmente e consuma crediti.


In [ ]:
from pathlib import Path
import subprocess
import sys

DRIVE_ROOT = Path("/content/drive/MyDrive/regulatory-insight-engine")
LOCAL_ROOT = Path("/content/regulatory-insight-engine-local")
ADAPTER_DIR = Path("/content/fiorellia_behavior_BLITZ_RELEASE_20260527")
HISTORY = Path("/content/fiorellia-runs/blitz_delivery_latest/blitz_app_history.jsonl")

summary_path = Path("/content/fiorellia-runs/blitz_delivery_latest/blitz_summary.json")
if not summary_path.exists():
    summary_path = DRIVE_ROOT / "fiorellia-runs" / "blitz_delivery_latest" / "blitz_summary.json"

print("Summary:", summary_path)
print(summary_path.read_text(encoding="utf-8")[:2000] if summary_path.exists() else "missing")

# Esegui solo quando vuoi tenere aperta la demo. Interrompi la cella per chiuderla.
cmd = [
    sys.executable,
    "-u",
    str(LOCAL_ROOT / "fiorellia_app_colab.py"),
    "--adapter-path",
    str(ADAPTER_DIR),
    "--history",
    str(HISTORY),
    "--max-new-tokens",
    "96",
    "--share",
]
print("+", " ".join(cmd))
subprocess.run(cmd, cwd=LOCAL_ROOT, check=False)
